# 03_lstm_run 

LSTM baseline — step-by-step training and evaluation. Owned by teammates. See src/lstm_baseline.py.

In [2]:
# Build vocabulary from training texts (whitespace split) and save to ../data/vocab.json
import sys
import json
from pathlib import Path
from collections import Counter
sys.path.insert(0, "..")
from src.data import load_splits
from src.utils import set_seed
set_seed(42)

train_df, val_df, test_df = load_splits()
texts = train_df["text"].astype(str).tolist()

# Vocab hyperparameters (tweakable)
min_freq = 2
max_vocab = 30000  # include special tokens in this limit
PAD_TOKEN = "<pad>"
UNK_TOKEN = "<unk>"
SPECIAL_TOKENS = [PAD_TOKEN, UNK_TOKEN]

# Count tokens using simple whitespace splitting on training texts only
ctr = Counter()
for t in texts:
    words = str(t).lower().split()
    ctr.update(words)

# Keep tokens meeting min_freq, then top-N
tokens = [tok for tok, cnt in ctr.most_common() if cnt >= min_freq]
if max_vocab:
    tokens = tokens[: max_vocab - len(SPECIAL_TOKENS)]

# Build mappings (pad=0, unk=1)
word2idx = {tok: idx for idx, tok in enumerate(SPECIAL_TOKENS + tokens)}
idx2word = {idx: word for word, idx in word2idx.items()}

# Save vocab to disk for reuse (word2idx mapping) — ensure correct path from notebook
vocab_path = Path('..') / 'data' / 'vocab.json'
vocab_path.parent.mkdir(parents=True, exist_ok=True)
with open(vocab_path, "w", encoding="utf8") as f:
    json.dump({"word2idx": word2idx}, f, ensure_ascii=False)

# Quick stats
total_tokens = sum(ctr.values())
vocab_size = len(word2idx)
coverage_top_5k = sum(c for w, c in ctr.items() if w in set(dict(ctr.most_common(5000)).keys())) / total_tokens

print(f"Saved vocab ({vocab_size} tokens) to {vocab_path}")
print(f"Total training tokens: {total_tokens:,}")
print(f"Coverage of top-5k tokens: {coverage_top_5k:.3f}")

# Small helper for later cells (if needed)
_vocab = {"word2idx": word2idx, "idx2word": idx2word, "pad_token": PAD_TOKEN, "unk_token": UNK_TOKEN}


Saved vocab (10350 tokens) to ..\data\vocab.json
Total training tokens: 310,055
Coverage of top-5k tokens: 0.902


In [2]:
import json
import numpy as np
import torch
from pathlib import Path

ROOT = Path("..")  # adjust if you run from repo root
vocab_path = ROOT / "data" / "vocab.json"
glove_path = ROOT / "data" / "glove.6B.300d.txt"
out_path = ROOT / "data" / "glove_embeddings.pt"
emb_dim = 300

# load vocab
with open(vocab_path, "r", encoding="utf8") as f:
    word2idx = json.load(f)["word2idx"]
vocab_size = len(word2idx)

# load GloVe robustly
assert glove_path.exists(), f"Missing {glove_path}"
glove = {}
with open(glove_path, "r", encoding="utf8", errors="ignore") as fh:
    for i, ln in enumerate(fh):
        parts = ln.rstrip().split()
        if len(parts) != emb_dim + 1:
            continue
        word = parts[0]
        vec = np.array(parts[1:], dtype=np.float32)
        glove[word] = vec
        if (i+1) % 100000 == 0:
            print(f"Read {i+1} lines")

# init embedding matrix
rng = np.random.default_rng(42)
emb_matrix = rng.normal(loc=0.0, scale=0.1, size=(vocab_size, emb_dim)).astype(np.float32)
pad_idx = word2idx.get("<pad>", 0)
emb_matrix[pad_idx] = np.zeros((emb_dim,), dtype=np.float32)

# fill from GloVe (try exact then lowercase) — avoid `or` with numpy arrays
for word, idx in word2idx.items():
    vec = glove.get(word)
    if vec is None:
        vec = glove.get(word.lower())
    if vec is not None:
        emb_matrix[idx] = vec

out_path.parent.mkdir(parents=True, exist_ok=True)
emb_tensor = torch.from_numpy(emb_matrix)
torch.save(emb_tensor, out_path)

# Optional: create embedding layer
embedding = torch.nn.Embedding.from_pretrained(emb_tensor, freeze=False, padding_idx=pad_idx)
print("Embedding ready:", embedding.weight.shape)


Read 100000 lines
Read 200000 lines
Read 300000 lines
Read 400000 lines
Embedding ready: torch.Size([10350, 300])


In [ ]:
# Model and data definitions (BiLSTM, Dataset, DataLoaders)
import sys
from pathlib import Path
import json
import yaml
import time

import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

sys.path.insert(0, "..")
from src.data import load_splits, class_weights
from src.utils import set_seed
from src.eval import evaluate

set_seed(42)

# Paths and config
ROOT = Path("..")
cfg = yaml.safe_load(open(ROOT / "configs" / "lstm.yaml"))
print("Config loaded:", cfg["run_name"]) 

# Load vocab and embeddings
vocab_path = ROOT / "data" / "vocab.json"
assert vocab_path.exists(), f"Vocab not found: {vocab_path}"
word2idx = json.loads(vocab_path.read_text(encoding="utf8"))["word2idx"]
pad_idx = word2idx.get("<pad>", 0)

emb_file = ROOT / "data" / "glove_embeddings.pt"
if emb_file.exists():
    emb_tensor = torch.load(emb_file)
    print("Loaded prebuilt embeddings:", emb_tensor.shape)
else:
    raise FileNotFoundError(f"Embeddings not found at {emb_file}. Run the embedding cell first.")

# BiLSTM model
class BiLSTMModel(nn.Module):
    def __init__(self, emb_tensor, cfg, num_classes=4):
        super().__init__()
        self.embedding = nn.Embedding.from_pretrained(emb_tensor, freeze=cfg.get("freeze_embeddings", False), padding_idx=pad_idx)
        self.hidden = cfg["hidden_dim"]
        self.num_layers = cfg["num_layers"]
        self.bidirectional = cfg.get("bidirectional", True)
        self.lstm = nn.LSTM(
            input_size=emb_tensor.shape[1],
            hidden_size=self.hidden,
            num_layers=self.num_layers,
            batch_first=True,
            dropout=cfg.get("dropout", 0.0) if self.num_layers>1 else 0.0,
            bidirectional=self.bidirectional,
        )
        out_dim = self.hidden * (2 if self.bidirectional else 1)
        self.fc = nn.Linear(out_dim, num_classes)
        self.dropout = nn.Dropout(cfg.get("dropout", 0.0))

    def forward(self, input_ids, lengths):
        emb = self.embedding(input_ids)
        lengths_sorted, perm_idx = lengths.sort(descending=True)
        emb_sorted = emb[perm_idx]
        packed = nn.utils.rnn.pack_padded_sequence(emb_sorted, lengths_sorted.cpu(), batch_first=True, enforce_sorted=True)
        packed_out, (h_n, c_n) = self.lstm(packed)
        if self.bidirectional:
            forward = h_n[-2]
            backward = h_n[-1]
            h = torch.cat([forward, backward], dim=1)
        else:
            h = h_n[-1]
        _, unperm_idx = perm_idx.sort()
        h = h[unperm_idx]
        h = self.dropout(h)
        logits = self.fc(h)
        return logits

# Dataset + collate
class TextDataset(Dataset):
    def __init__(self, df, word2idx, max_len=128):
        self.texts = df["text"].astype(str).tolist()
        self.labels = df["label_id"].astype(int).tolist()
        self.word2idx = word2idx
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        toks = str(self.texts[idx]).lower().split()
        ids = [self.word2idx.get(t, self.word2idx.get("<unk>")) for t in toks]
        if len(ids) > self.max_len:
            ids = ids[:self.max_len]
        return ids, self.labels[idx]

def collate_batch(batch):
    ids_list, labels = zip(*batch)
    lengths = torch.tensor([len(x) for x in ids_list], dtype=torch.long)
    max_l = int(lengths.max()) if len(lengths)>0 else 0
    padded = torch.full((len(ids_list), max_l), pad_idx, dtype=torch.long)
    for i, seq in enumerate(ids_list):
        padded[i, :len(seq)] = torch.tensor(seq, dtype=torch.long)
    labels = torch.tensor(labels, dtype=torch.long)
    return padded, lengths, labels

# Prepare data loaders
train_df, val_df, test_df = load_splits()
train_ds = TextDataset(train_df, word2idx, max_len=cfg.get("max_length", 128))
val_ds   = TextDataset(val_df,   word2idx, max_len=cfg.get("max_length", 128))
test_ds  = TextDataset(test_df,  word2idx, max_len=cfg.get("max_length", 128))

batch_size = cfg.get("batch_size", 64)
train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True, collate_fn=collate_batch)
val_loader   = DataLoader(val_ds,   batch_size=batch_size, shuffle=False, collate_fn=collate_batch)
test_loader  = DataLoader(test_ds,  batch_size=batch_size, shuffle=False, collate_fn=collate_batch)

# Build model, optimizer, loss
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = BiLSTMModel(emb_tensor, cfg).to(device)
opt = torch.optim.AdamW(model.parameters(), lr=cfg.get("learning_rate", 1e-3))
weights = class_weights(train_df).to(device)
criterion = nn.CrossEntropyLoss(weight=weights)

print('Model and dataloaders ready')


In [ ]:
# Training loop with early stopping and test evaluation
import time
from sklearn.metrics import f1_score
import numpy as np

best_val = -1.0
patience = cfg.get("early_stopping_patience", 3)
no_improve = 0
num_epochs = cfg.get("num_epochs", 10)
out_dir = ROOT / cfg.get("output_dir", "results/runs/lstm")
out_dir.mkdir(parents=True, exist_ok=True)
best_ckpt = out_dir / "best.pt"

for epoch in range(1, num_epochs+1):
    model.train()
    t0 = time.time()
    running_loss = 0.0
    for batch in train_loader:
        input_ids, lengths, labels = batch
        input_ids = input_ids.to(device)
        lengths = lengths.to(device)
        labels = labels.to(device)
        opt.zero_grad()
        logits = model(input_ids, lengths)
        loss = criterion(logits, labels)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0)
        opt.step()
        running_loss += loss.item() * input_ids.size(0)
    train_loss = running_loss / len(train_ds)

    # validation
    model.eval()
    all_preds = []
    all_labels = []
    with torch.no_grad():
        for batch in val_loader:
            input_ids, lengths, labels = batch
            input_ids = input_ids.to(device)
            lengths = lengths.to(device)
            labels = labels.to(device)
            logits = model(input_ids, lengths)
            preds = logits.argmax(dim=1).cpu().numpy()
            all_preds.extend(preds.tolist())
            all_labels.extend(labels.cpu().numpy().tolist())
    val_f1 = f1_score(all_labels, all_preds, average="macro")
    elapsed = time.time()-t0
    print(f"Epoch {epoch}/{num_epochs} — train_loss={train_loss:.4f} val_macro_f1={val_f1:.4f} time={elapsed:.1f}s")

    # checkpoint
    if val_f1 > best_val:
        best_val = val_f1
        no_improve = 0
        torch.save({"model_state": model.state_dict(), "opt_state": opt.state_dict(), "epoch": epoch, "val_f1": val_f1}, best_ckpt)
        print("Saved best checkpoint", best_ckpt)
    else:
        no_improve += 1
        print(f"No improvement ({no_improve}/{patience})")
    if no_improve >= patience:
        print("Early stopping")
        break

# Load best model and run on test set
ckpt = torch.load(best_ckpt, map_location=device)
model.load_state_dict(ckpt["model_state"]) 
model.to(device)

all_preds = []
all_labels = []
with torch.no_grad():
    for batch in test_loader:
        input_ids, lengths, labels = batch
        input_ids = input_ids.to(device)
        lengths = lengths.to(device)
        logits = model(input_ids, lengths)
        preds = logits.argmax(dim=1).cpu().numpy()
        all_preds.extend(preds.tolist())
        all_labels.extend(labels.numpy().tolist())

# Call repo evaluate() to save metrics and confusion matrix
evaluate(preds=np.array(all_preds), labels=np.array(all_labels), df=test_df, run_name=cfg.get("run_name","lstm"), output_dir=str(out_dir), split="test")
print("Finished evaluation. Results saved to", out_dir)
